# Load the warehouse: `warehouse.db`

This notebook and warehouse_schema.md were both initially generated by llm prompt 4 - and then edited/reviewed as needed.

This notebook loads the validated tables from `data_cleaning.ipynb` into a SQLite star schema:

```
              dim_date
                 |
dim_store --- fact_sales --- dim_product
```

**Input:** `../data/validated/stores.csv`, `products.csv`, `transactions.csv`
**Output:** `../data/warehouse.db`, rebuilt from scratch on every run
**Schema documentation:** `../docs/warehouse_schema.md`

Run `data_cleaning.ipynb` first, then use *Kernel → Restart & Run All* here. You can also run it headless:
```
jupyter nbconvert --to notebook --execute --inplace load_warehouse.ipynb
```
The database file is deleted and recreated on every run, so the result is the same each time. The tables are built in memory first and checked, and only then is the database written. A failed check stops the notebook (`STRICT = True`), so an existing `warehouse.db` is never replaced with bad data.

## 1. Setup

In [1]:
from pathlib import Path
import re
import sqlite3

import pandas as pd
from IPython.display import display

VALIDATED_DIR = Path("../data/validated")
DB_PATH = Path("../output/warehouse.db")
STRICT = True


def check(ok, message):
    if ok:
        print(f"PASS  {message}")
        return True
    if STRICT:
        raise AssertionError(f"CHECK FAILED: {message}\nSTRICT = True, so the notebook stops before "
                             "writing the database. Set STRICT = False to continue with a warning.")
    print(f"WARN  {message}")
    return False


def info(message):
    print(f"INFO  {message}")


def to_flag(s, label):
    """'True'/'False' text from the validated CSVs -> 1/0."""
    out = s.map({"True": 1, "False": 0})
    check(out.notna().all(), f"{label} holds only True/False")
    return out.astype("int64")


def blank_to_none(s):
    return s.where(s.str.strip() != "", None)

### Load the validated tables
Everything is read as text, so IDs and zip codes come in exactly as they were written. Each column is then converted to its warehouse type below.

In [2]:
read = lambda f: pd.read_csv(VALIDATED_DIR / f, dtype=str, keep_default_na=False)
stores, products, transactions = read("stores.csv"), read("products.csv"), read("transactions.csv")

for name, df in {"stores": stores, "products": products, "transactions": transactions}.items():
    print(f"{name:13s} {len(df):>6,} rows")

stores            19 rows
products          32 rows
transactions     485 rows


## 2. Build the dimensions

Each dimension gets an integer surrogate key (`*_key`), numbered in order of the natural ID. The natural ID (`store_id`, `product_id`) is kept as a unique column, so tables can still be joined back to the source data.

### dim_store

In [3]:
dim_store = stores.sort_values("store_id").reset_index(drop=True)
check(dim_store["store_id"].is_unique, "store_id is unique")
dim_store.insert(0, "store_key", range(1, len(dim_store) + 1))
dim_store["zip_code_invalid"] = to_flag(dim_store["zip_code_invalid"], "zip_code_invalid")
dim_store["store_details_missing"] = to_flag(dim_store["store_details_missing"], "store_details_missing")
for col in ("zip_code", "opened_date"):
    dim_store[col] = blank_to_none(dim_store[col])
dim_store = dim_store[["store_key", "store_id", "store_name", "city", "state", "zip_code",
                       "zip_code_invalid", "region", "opened_date", "store_details_missing"]]
info(f"{int(dim_store['store_details_missing'].sum())} placeholder stores (in transactions, not in the stores file)")
display(dim_store.head())

PASS  store_id is unique
PASS  zip_code_invalid holds only True/False
PASS  store_details_missing holds only True/False
INFO  4 placeholder stores (in transactions, not in the stores file)


,store_key,store_id,store_name,city,state,zip_code,zip_code_invalid,region,opened_date,store_details_missing
0,1,S001,Eastview Mall,Victor,NY,14564,0,Northeast,2010-03-15,0
1,2,S002,Marketplace Mall,Rochester,NY,14623,0,Northeast,2008-11-01,0
2,3,S003,Greece Ridge Center,Greece,NY,0938,1,Northeast,2012-06-20,0
3,4,S004,Crossroads Center,St. Cloud,MN,56301,0,Midwest,2015-09-10,0
4,5,S005,Mall of America,Bloomington,MN,55425,0,Midwest,2011-04-07,0


### dim_product
`product_name` is left out, because it is just the `product_id` with "product" at the start as a prefix. The check below confirms that is true for every row before the column is dropped.

In [4]:
norm = lambda s: s.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
name_is_prefixed_id = norm(products["product_name"]) == "product" + norm(products["product_id"])
if not name_is_prefixed_id.all():
    display(products[~name_is_prefixed_id])
check(name_is_prefixed_id.all(), "every product_name is 'product' + product_id")

dim_product = products.sort_values("product_id").reset_index(drop=True)
check(dim_product["product_id"].is_unique, "product_id is unique")
dim_product.insert(0, "product_key", range(1, len(dim_product) + 1))
dim_product["product_details_missing"] = to_flag(dim_product["product_details_missing"], "product_details_missing")
dim_product["catalog_unit_price"] = pd.to_numeric(blank_to_none(dim_product["unit_price"]), errors="raise").round(2)
real = dim_product["product_details_missing"] == 0
check((dim_product.loc[real, "catalog_unit_price"] > 0).all(), "every real product has a catalog price above 0")
check(dim_product.loc[~real, "catalog_unit_price"].isna().all(), "placeholder products have no catalog price")
dim_product = dim_product[["product_key", "product_id", "category", "supplier_id", "catalog_unit_price",
                           "product_details_missing"]]
info(f"{int((~real).sum())} placeholder products (in transactions, not in the products file)")
display(dim_product.head())

PASS  every product_name is 'product' + product_id
PASS  product_id is unique
PASS  product_details_missing holds only True/False
PASS  every real product has a catalog price above 0
PASS  placeholder products have no catalog price
INFO  2 placeholder products (in transactions, not in the products file)


,product_key,product_id,category,supplier_id,catalog_unit_price,product_details_missing
0,1,P001,Electronics,SUP001,157.39,0
1,2,P002,Apparel,SUP002,147.38,0
2,3,P003,Home & Garden,SUP003,134.20,0
3,4,P004,Food & Beverage,SUP004,10.53,0
4,5,P005,Office Supplies,SUP005,141.61,0


### dim_date
There is one row for every calendar day from the first transaction date to the last, including days with no sales, so reports by day show those days as zero instead of skipping them. `date_key` is the date as an integer in `YYYYMMDD` form.

In [5]:
tx_dates = pd.to_datetime(transactions["transaction_date"], format="%Y-%m-%d", errors="raise")
days = pd.date_range(tx_dates.min(), tx_dates.max(), freq="D")
dim_date = pd.DataFrame({
    "date_key": days.strftime("%Y%m%d").astype("int64"),
    "full_date": days.strftime("%Y-%m-%d"),
    "year": days.year,
    "quarter": days.quarter,
    "month": days.month,
    "month_name": days.strftime("%B"),
    "day_of_month": days.day,
    "day_of_week": days.dayofweek + 1,          # 1 = Monday ... 7 = Sunday
    "day_name": days.strftime("%A"),
    "iso_week": days.isocalendar().week.to_numpy().astype("int64"),
    "is_weekend": (days.dayofweek >= 5).astype("int64"),
})
info(f"dim_date: {len(dim_date)} days, {dim_date['full_date'].iloc[0]} to {dim_date['full_date'].iloc[-1]}")
display(dim_date.head())

INFO  dim_date: 115 days, 2026-03-05 to 2026-06-27


,date_key,full_date,year,quarter,month,month_name,day_of_month,day_of_week,day_name,iso_week,is_weekend
0,20260305,2026-03-05,2026,1,3,March,5,4,Thursday,10,0
1,20260306,2026-03-06,2026,1,3,March,6,5,Friday,10,0
2,20260307,2026-03-07,2026,1,3,March,7,6,Saturday,10,1
3,20260308,2026-03-08,2026,1,3,March,8,7,Sunday,10,1
4,20260309,2026-03-09,2026,1,3,March,9,1,Monday,11,0


## 3. Build fact_sales
There is one row per transaction. `transaction_id` and `customer_id` stay on the fact table as degenerate dimensions, meaning IDs with no dimension table of their own. The return flags from cleaning are kept, so returns and the sales they reverse remain in the data but can be filtered out with `exclude_from_metrics`.

In [6]:
fact = transactions.copy()
check(fact["transaction_id"].is_unique, "transaction_id is unique")

fact["date_key"] = tx_dates.dt.strftime("%Y%m%d").astype("int64")
fact = fact.merge(dim_store[["store_id", "store_key"]], on="store_id", how="left")
fact = fact.merge(dim_product[["product_id", "product_key"]], on="product_id", how="left")
for key, natural in (("store_key", "store_id"), ("product_key", "product_id")):
    missing = fact[fact[key].isna()]
    if len(missing):
        display(missing)
    check(missing.empty, f"every transaction's {natural} exists in the dimension")

fact["quantity"] = pd.to_numeric(fact["quantity"], errors="raise").astype("int64")
fact["unit_price"] = pd.to_numeric(fact["unit_price"], errors="raise").round(2)
fact["total_amount"] = pd.to_numeric(fact["total_amount"], errors="raise").round(2)
for col in ("is_return", "is_returned_sale", "exclude_from_metrics"):
    fact[col] = to_flag(fact[col], col)
fact["customer_id"] = blank_to_none(fact["customer_id"])
fact["matched_transaction_id"] = blank_to_none(fact["matched_transaction_id"])

check((fact["quantity"] != 0).all(), "no zero-quantity rows")
check(((fact["quantity"] < 0) == (fact["is_return"] == 1)).all(), "is_return matches negative quantity")
pairs_ok = fact["matched_transaction_id"].dropna().isin(fact["transaction_id"]).all()
check(pairs_ok, "every matched_transaction_id points to a transaction in the fact table")

fact_sales = fact[["transaction_id", "date_key", "store_key", "product_key", "customer_id", "quantity",
                   "unit_price", "total_amount", "is_return", "is_returned_sale",
                   "matched_transaction_id", "exclude_from_metrics"]].astype({"store_key": "int64",
                                                                              "product_key": "int64"})
off_catalog = fact.merge(dim_product[["product_id", "catalog_unit_price"]], on="product_id")
off_catalog = (off_catalog["unit_price"] != off_catalog["catalog_unit_price"]).sum()
info(f"{off_catalog} transactions with a unit_price different from the catalog price (kept as transacted)")
display(fact_sales.head())

PASS  transaction_id is unique
PASS  every transaction's store_id exists in the dimension
PASS  every transaction's product_id exists in the dimension
PASS  is_return holds only True/False
PASS  is_returned_sale holds only True/False
PASS  exclude_from_metrics holds only True/False
PASS  no zero-quantity rows
PASS  is_return matches negative quantity
PASS  every matched_transaction_id points to a transaction in the fact table
INFO  3 transactions with a unit_price different from the catalog price (kept as transacted)


,transaction_id,date_key,store_key,product_key,customer_id,quantity,unit_price,total_amount,is_return,is_returned_sale,matched_transaction_id,exclude_from_metrics
0,TXN10179,20260530,11,24,CUST0247,2,196.20,392.40,0,0,None,0
1,TXN10209,20260423,15,27,None,4,195.34,781.36,0,0,None,0
2,TXN10385,20260311,14,30,CUST0059,4,76.48,305.92,0,0,None,0
3,TXN10130,20260417,11,10,CUST0060,5,237.08,1185.40,0,0,None,0
4,TXN20016,20260525,7,30,CUST0236,-2,76.48,-152.96,1,0,TXN10081,1


## 4. Write the database
The schema is created with primary keys, foreign keys and `CHECK` constraints, so SQLite itself refuses bad rows. `matched_transaction_id` is a foreign key back to `fact_sales`, set as deferred: a return and its sale point at each other, so the check runs when the whole load is committed.

In [7]:
DDL = """
PRAGMA foreign_keys = ON;

CREATE TABLE dim_date (
    date_key      INTEGER PRIMARY KEY,                 -- YYYYMMDD
    full_date     TEXT    NOT NULL UNIQUE,             -- YYYY-MM-DD
    year          INTEGER NOT NULL,
    quarter       INTEGER NOT NULL CHECK (quarter BETWEEN 1 AND 4),
    month         INTEGER NOT NULL CHECK (month BETWEEN 1 AND 12),
    month_name    TEXT    NOT NULL,
    day_of_month  INTEGER NOT NULL CHECK (day_of_month BETWEEN 1 AND 31),
    day_of_week   INTEGER NOT NULL CHECK (day_of_week BETWEEN 1 AND 7),   -- 1 = Monday
    day_name      TEXT    NOT NULL,
    iso_week      INTEGER NOT NULL CHECK (iso_week BETWEEN 1 AND 53),
    is_weekend    INTEGER NOT NULL CHECK (is_weekend IN (0, 1))
);

CREATE TABLE dim_store (
    store_key         INTEGER PRIMARY KEY,
    store_id          TEXT    NOT NULL UNIQUE,
    store_name        TEXT    NOT NULL,
    city              TEXT,
    state             TEXT,
    zip_code          TEXT,                            -- text: keeps leading zeros
    zip_code_invalid  INTEGER NOT NULL CHECK (zip_code_invalid IN (0, 1)),
    region            TEXT    NOT NULL,
    opened_date       TEXT,
    store_details_missing INTEGER NOT NULL CHECK (store_details_missing IN (0, 1))  -- 1 = placeholder
);

CREATE TABLE dim_product (
    product_key         INTEGER PRIMARY KEY,
    product_id          TEXT    NOT NULL UNIQUE,
    category            TEXT    NOT NULL,
    supplier_id         TEXT    NOT NULL,
    catalog_unit_price  REAL,                          -- NULL only for placeholder products
    product_details_missing INTEGER NOT NULL CHECK (product_details_missing IN (0, 1)),  -- 1 = placeholder
    CHECK ((product_details_missing = 0 AND catalog_unit_price > 0)
        OR (product_details_missing = 1 AND catalog_unit_price IS NULL))
);

CREATE TABLE fact_sales (
    transaction_id          TEXT    PRIMARY KEY,
    date_key                INTEGER NOT NULL REFERENCES dim_date (date_key),
    store_key               INTEGER NOT NULL REFERENCES dim_store (store_key),
    product_key             INTEGER NOT NULL REFERENCES dim_product (product_key),
    customer_id             TEXT,
    quantity                INTEGER NOT NULL CHECK (quantity <> 0),
    unit_price              REAL    NOT NULL,
    total_amount            REAL    NOT NULL,
    is_return               INTEGER NOT NULL CHECK (is_return IN (0, 1)),
    is_returned_sale        INTEGER NOT NULL CHECK (is_returned_sale IN (0, 1)),
    matched_transaction_id  TEXT    REFERENCES fact_sales (transaction_id) DEFERRABLE INITIALLY DEFERRED,
    exclude_from_metrics    INTEGER NOT NULL CHECK (exclude_from_metrics IN (0, 1)),
    CHECK (exclude_from_metrics = (is_return OR is_returned_sale))
);

CREATE INDEX ix_fact_sales_date    ON fact_sales (date_key);
CREATE INDEX ix_fact_sales_store   ON fact_sales (store_key);
CREATE INDEX ix_fact_sales_product ON fact_sales (product_key);

-- Sales to use for revenue and other metrics: returns and the sales they reverse are left out.
CREATE VIEW v_sales_reportable AS
SELECT * FROM fact_sales WHERE exclude_from_metrics = 0;
"""

TABLES = {"dim_date": dim_date, "dim_store": dim_store, "dim_product": dim_product, "fact_sales": fact_sales}


def insert(con, table, df):
    rows = [tuple(None if pd.isna(v) else (v.item() if hasattr(v, "item") else v) for v in r)
            for r in df.itertuples(index=False)]
    cols = ", ".join(df.columns)
    marks = ", ".join("?" * len(df.columns))
    con.executemany(f"INSERT INTO {table} ({cols}) VALUES ({marks})", rows)


DB_PATH.parent.mkdir(parents=True, exist_ok=True)
tmp_path = DB_PATH.with_suffix(".db.tmp")
tmp_path.unlink(missing_ok=True)

con = sqlite3.connect(tmp_path)
try:
    con.executescript(DDL)
    with con:  # one transaction: deferred foreign keys are checked on commit
        for table, df in TABLES.items():
            insert(con, table, df)
            print(f"loaded {table:12s} {len(df):>6,} rows")
finally:
    con.close()

tmp_path.replace(DB_PATH)  # swap in the new database only once it loaded cleanly
print(f"wrote {DB_PATH.resolve()}")

loaded dim_date        115 rows
loaded dim_store        19 rows
loaded dim_product      32 rows
loaded fact_sales      485 rows
wrote C:\Users\Anthony Viola\Desktop\data-engineer-code-challenge\data-engineer-code-challenge-viola\output\warehouse.db


## 5. Verify the database

In [8]:
con = sqlite3.connect(DB_PATH)
q = lambda sql: pd.read_sql_query(sql, con)

for table, df in TABLES.items():
    n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    check(n == len(df), f"{table}: {n:,} rows in the database")

check(con.execute("PRAGMA integrity_check").fetchone()[0] == "ok", "SQLite integrity check")
check(not con.execute("PRAGMA foreign_key_check").fetchall(), "no foreign key violations")

db_total = con.execute("SELECT ROUND(SUM(total_amount), 2) FROM fact_sales").fetchone()[0]
csv_total = round(pd.to_numeric(transactions["total_amount"]).sum(), 2)
check(abs(db_total - csv_total) < 0.005, f"total_amount in the database ({db_total:,.2f}) matches the validated CSV")

display(q("""
    SELECT d.month_name,
           ROUND(SUM(CASE WHEN f.exclude_from_metrics = 0 THEN f.total_amount END), 2) AS reportable_revenue,
           SUM(f.is_return)                                                          AS returns,
           COUNT(*)                                                                  AS transactions
    FROM fact_sales f JOIN dim_date d USING (date_key)
    GROUP BY d.year, d.month ORDER BY d.year, d.month
"""))
con.close()

PASS  dim_date: 115 rows in the database
PASS  dim_store: 19 rows in the database
PASS  dim_product: 32 rows in the database
PASS  fact_sales: 485 rows in the database
PASS  SQLite integrity check
PASS  no foreign key violations
PASS  total_amount in the database (160,827.03) matches the validated CSV


,month_name,reportable_revenue,returns,transactions
0,March,41695.27,4,125
1,April,53967.19,13,155
2,May,62796.69,13,197
3,June,2367.88,0,8
